# 09 — IBM Environment, Authentication and Calibration Capture

This notebook is the **pre-flight gate** before real adaptive-QEM experiments. It verifies the installed software stack, IBM Quantum authentication, `ibm_kingston` availability, backend configuration, coupling information, and calibration data. **No experiment is submitted.**

IBM's current documentation recommends Qiskit 2.5.2+ and qiskit-ibm-runtime 0.47.0+ for the current Sampler workflow. `ibm_kingston` is currently listed as a 156-qubit Heron r2 processor. 


In [ ]:
import sys, json, platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

import qiskit
print("Qiskit:", qiskit.__version__)

import qiskit_ibm_runtime
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)


## 1. Version gate

The experiment should not proceed silently on an untested API combination. The values below are a compatibility warning, not a claim that older versions can never work.


In [ ]:
from packaging.version import Version

QISKIT_MIN = Version("2.5.2")
RUNTIME_MIN = Version("0.47.0")

qv = Version(qiskit.__version__)
rv = Version(qiskit_ibm_runtime.__version__)

print("Qiskit meets current documented recommendation:", qv >= QISKIT_MIN)
print("Runtime meets current documented recommendation:", rv >= RUNTIME_MIN)

if qv < QISKIT_MIN or rv < RUNTIME_MIN:
    print("WARNING: validate this environment before hardware submission.")


## 2. IBM Quantum authentication

Use the locally configured IBM Quantum credentials. Do not place an API token in this notebook.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
print("IBM Quantum Runtime service initialized successfully.")


## 3. Kingston backend check

In [ ]:
BACKEND_NAME = "ibm_kingston"
backend = service.backend(BACKEND_NAME)
status = backend.status()

print("Name:", backend.name)
print("Operational:", status.operational)
print("Pending jobs:", status.pending_jobs)
print("Status message:", status.status_msg)
print("Num qubits:", backend.num_qubits)

assert backend.num_qubits == 156, f"Unexpected qubit count: {backend.num_qubits}"


## 4. Backend configuration and processor metadata

In [ ]:
for attr in ["backend_version", "num_qubits", "basis_gates", "coupling_map"]:
    try:
        value = getattr(backend, attr)
        if callable(value):
            value = value()
        if attr == "coupling_map":
            try:
                print(attr, ":", len(value), "connections")
            except Exception:
                print(attr, ":", value)
        else:
            print(attr, ":", value)
    except Exception as exc:
        print(attr, ": unavailable ->", exc)


## 5. Calibration capture

The exact backend property schema can differ across Runtime/Qiskit versions. Missing fields are preserved as `None`; they are never replaced with assumed values.


In [ ]:
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

props = None
try:
    props = backend.properties()
except Exception as exc:
    print("backend.properties() unavailable:", exc)

import pandas as pd

rows = []
if props is not None:
    for q in range(backend.num_qubits):
        row = {
            "qubit": q,
            "t1_s": None,
            "t2_s": None,
            "readout_error": None,
            "prob_meas0_prep1": None,
            "prob_meas1_prep0": None,
        }

        for name, target in [
            ("T1", "t1_s"),
            ("T2", "t2_s"),
            ("readout_error", "readout_error"),
            ("prob_meas0_prep1", "prob_meas0_prep1"),
            ("prob_meas1_prep0", "prob_meas1_prep0"),
        ]:
            try:
                value = props.qubit_property(name, q)
                if isinstance(value, list) and value:
                    value = value[0]
                row[target] = safe_float(value)
            except Exception:
                pass

        rows.append(row)

cal_df = pd.DataFrame(rows)
display(cal_df.head())


## 6. Calibration summary

In [ ]:
summary = {}

for col in ["t1_s", "t2_s", "readout_error",
            "prob_meas0_prep1", "prob_meas1_prep0"]:
    if col in cal_df:
        s = pd.to_numeric(cal_df[col], errors="coerce").dropna()
        if len(s):
            summary[col] = {
                "n": int(len(s)),
                "mean": float(s.mean()),
                "std": float(s.std()),
                "min": float(s.min()),
                "max": float(s.max()),
            }

display(pd.DataFrame(summary).T)


## 7. Coupling map snapshot

In [ ]:
coupling = None

try:
    coupling = backend.coupling_map
    if callable(coupling):
        coupling = coupling()
except Exception:
    try:
        coupling = backend.configuration().coupling_map
    except Exception:
        coupling = []

print("Coupling edges:", len(coupling) if coupling is not None else 0)


## 8. Save calibration snapshot

This snapshot becomes part of the experiment provenance. Re-run this notebook immediately before the hardware campaign and retain the resulting files.


In [ ]:
OUT = Path("data/calibration")
OUT.mkdir(parents=True, exist_ok=True)

cal_df.to_csv(OUT / "ibm_kingston_calibration_snapshot.csv", index=False)

if coupling is not None:
    (OUT / "ibm_kingston_coupling_map.json").write_text(
        json.dumps(coupling), encoding="utf-8"
    )

manifest = {
    "backend": backend.name,
    "backend_version": str(getattr(backend, "backend_version", "unknown")),
    "num_qubits": int(backend.num_qubits),
    "operational": bool(status.operational),
    "pending_jobs": int(status.pending_jobs),
    "software": {
        "python": sys.version,
        "qiskit": qiskit.__version__,
        "qiskit_ibm_runtime": qiskit_ibm_runtime.__version__,
    },
}

(OUT / "ibm_kingston_calibration_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Saved calibration snapshot to:", OUT.resolve())


## Pre-flight decision

Proceed to Notebook 08 only if authentication succeeds, the intended backend is operational, the measured qubit count is correct, and the installed software stack has been validated.

No hardware jobs are submitted by this notebook.
